In [ ]:
import pandas as pd
import numpy as np
import re
import os

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/vidhitts/ghjgfhj/amazon_products_sales_data_uncleaned.csv")
df.head()

In [ ]:
df1 = df.copy()

In [ ]:
df1.drop_duplicates(inplace=True)

In [ ]:
df1['price_on_variant'] = df1['price_on_variant'].str.split(':').str.get(1)

In [ ]:
df1.loc[~df1['price_on_variant'].str.contains(r'\$', na=False), 'price_on_variant'] = np.nan

In [ ]:
df1['price_on_variant'] = df1['price_on_variant'].str.strip().str.split(' ').str.get(0)

In [ ]:
df1['current/discounted_price'] = df1['current/discounted_price'].fillna(df1['price_on_variant'])

In [ ]:

df1['current/discounted_price'] = (
    df1['current/discounted_price']
    .astype(str)
    .str.replace(r'\$', '', regex=True)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

In [ ]:
df1['listed_price'] = (
    df1['listed_price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace('No Discount', np.nan)

In [ ]:
df1['listed_price'] = df1['listed_price'].fillna(df1['current/discounted_price'])
df1['listed_price'] = df1['listed_price'].astype(float)

In [ ]:

df1.drop(columns=['price_on_variant'], inplace=True)

In [ ]:
df1['discount_percentage'] = (
    (df1['listed_price'] - df1['current/discounted_price'])
    / df1['listed_price'] * 100
).round(2)

In [ ]:
df1['discount_percentage'] = df1['discount_percentage'].clip(lower=0)

In [ ]:

df1['rating'] = (
    df1['rating']
    .str.replace('out of 5 stars', '', regex=False)
    .str.strip()
    .astype(float)
)

In [ ]:
df1['number_of_reviews'] = (
    df1['number_of_reviews']
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

In [ ]:

def parse_bought(val):
    """Convert '20K+ bought in past month' → 20000, '200+ …' → 200."""
    if pd.isna(val):
        return pd.NA
    val = str(val).replace('+ bought in past month', '').replace('+', '').strip()
    if 'K' in val:
        try:
            return int(float(val.replace('K', '')) * 1000)
        except ValueError:
            return pd.NA
    try:
        return int(val)
    except ValueError:
        return pd.NA

df1['bought_in_last_month'] = df1['bought_in_last_month'].apply(parse_bought).astype('Int64')

In [ ]:
def clean_best_seller(val):
    if pd.isna(val) or val == 'No Badge':
        return 'No Badge'
    if 'Best Seller' in str(val):
        return 'Best Seller'
    if "Amazon's" in str(val):
        return "Amazon's Choice"
    if 'Limited time deal' in str(val):
        return 'Limited time deal'
    return 'No Badge'   # "Save 30%", "Ends in X hours", etc. → artefacts

df1['is_best_seller'] = df1['is_best_seller'].apply(clean_best_seller)

In [ ]:
df1['is_sponsored'] = df1['is_sponsored'].map({'Sponsored': True, 'Organic': False})

In [ ]:
df1['has_coupon']   = df1['is_couponed'].ne('No Coupon')
df1['coupon_value'] = df1['is_couponed'].where(df1['has_coupon'], np.nan)
df1.drop(columns=['is_couponed'], inplace=True)

In [ ]:

df1['buy_box_availability'] = df1['buy_box_availability'].notna()

In [ ]:

collected_year = pd.to_datetime(df1['collected_at'].dropna().iloc[0]).year

In [ ]:
df1['delivery_details'] = df1['delivery_details'].str.extract(
    r'(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)?,?\s*(\w+\s+\d{1,2})'
)
df1['delivery_details'] = pd.to_datetime(
    df1['delivery_details'] + f' {collected_year}', errors='coerce'
)

In [ ]:
df1['collected_at'] = pd.to_datetime(df1['collected_at'])

In [ ]:
CATEGORY_KEYWORDS = {
    'Laptops':            ['laptop','notebook','macbook','chromebook','ultrabook',
                           'acer','asus','dell','lenovo','hp','core','intel',
                           'ryzen','surface','thinkpad','ideapad'],
    'Phones':             ['phone','iphone','smartphone','samsung','android','galaxy',
                           'pixel','oneplus','xiaomi','oppo','realme','huawei',
                           'vivo','nokia','motorola'],
    'Headphones':         ['headphone','headset','earphone','earbuds','airpods',
                           'beats','sony wh','wireless buds','neckband'],
    'Chargers & Cables':  ['charger','charging','cable','adapter','dock','usb c',
                           'type c','lightning','power adapter','usb cable'],
    'Cameras':            ['camera','dslr','mirrorless','canon','nikon','gopro',
                           'instax','webcam','camcorder','security camera'],
    'Storage':            ['ssd','hard drive','memory card','flash drive','pendrive',
                           'hdd','storage','micro sd','sd card'],
    'Smart Home':         ['alexa','echo','smart plug','smart bulb','smart home',
                           'nest','homekit','smart switch'],
    'TV & Display':       ['monitor','display','tv','screen','projector','oled',
                           'led','curved monitor','uhd','4k'],
    'Power & Batteries':  ['battery','power bank','rechargeable','aa','aaa',
                           'portable power','cell'],
    'Networking':         ['wifi','router','modem','ethernet','access point',
                           'mesh','network switch'],
    'Wearables':          ['smartwatch','fitness band','fitbit','watch',
                           'garmin','amazfit'],
    'Speakers':           ['speaker','soundbar','subwoofer','bluetooth speaker',
                           'party speaker','home theater'],
    'Printers & Scanners':['printer','scanner','inkjet','laserjet',
                           'photocopier','all in one printer'],
    'Gaming':             ['gaming console','playstation','ps5','ps4','xbox',
                           'nintendo','joystick','controller','gaming mouse',
                           'gaming keyboard','gaming chair'],
    'Other Electronics':  []
}

def assign_category(title):
    t = re.sub(r'[^a-z0-9\s]', '', title.lower())
    for cat, kws in CATEGORY_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return cat
    return 'Other Electronics'

df1['category'] = df1['title'].apply(assign_category)

In [ ]:
df1.rename(columns={
    'title':                'product_title',
    'rating':               'product_rating',
    'number_of_reviews':    'total_reviews',
    'bought_in_last_month': 'purchased_last_month',
    'current/discounted_price': 'discounted_price',
    'listed_price':         'original_price',
    'delivery_details':     'delivery_date',
    'sustainability_badges':'sustainability_tags',
    'image_url':            'product_image_url',
    'product_url':          'product_page_url',
    'collected_at':         'data_collected_at',
    'category':             'product_category',
}, inplace=True)

In [ ]:
df1.to_csv("amazon_electronics_cleaned(111).csv", index=False)